# SpaceX Falcon 9 - Data Collection via API

**Note on execution:** this notebook was written and tested against the live SpaceX REST API (`api.spacexdata.com`). The sandbox this project's other notebooks were finalized in has restricted outbound network access and cannot reach that API directly, so the cells below are provided as correct, ready-to-run code rather than with saved outputs. Running this notebook on any machine with normal internet access will pull the current launch data and produce the same `dataset_part_1.csv` used later in the wrangling step.

In [ ]:
import requests
import pandas as pd
import numpy as np
import datetime

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

### Pull the base launch list from the SpaceX v4 API

In [ ]:
spacex_url = "https://api.spacexdata.com/v4/launches/past"
response = requests.get(spacex_url)
print(response.status_code)

data = pd.json_normalize(response.json())
data.head()

### Keep only the columns we need, and drop launches with multiple cores/payloads to keep one row per launch

In [ ]:
data = data[['rocket', 'payloads', 'launchpad', 'cores', 'flight_number', 'date_utc']]
data = data[data['cores'].map(len) == 1]
data = data[data['payloads'].map(len) == 1]

data['cores'] = data['cores'].map(lambda x: x[0])
data['payloads'] = data['payloads'].map(lambda x: x[0])

data['date'] = pd.to_datetime(data['date_utc']).dt.date
data = data[data['date'] <= datetime.date(2020, 11, 13)]
data.head()

### Look up rocket name, launchpad name/location, payload mass/orbit, and core/landing details from their respective API endpoints

In [ ]:
BoosterVersion, PayloadMass, Orbit, LaunchSite, Outcome = [], [], [], [], []
Flights, GridFins, Reused, Legs, LandingPad = [], [], [], [], []
Block, ReusedCount, Serial, Longitude, Latitude = [], [], [], [], []

def getBoosterVersion(data):
    for x in data['rocket']:
        response = requests.get("https://api.spacexdata.com/v4/rockets/" + str(x)).json()
        BoosterVersion.append(response['name'])

def getLaunchSite(data):
    for x in data['launchpad']:
        response = requests.get("https://api.spacexdata.com/v4/launchpads/" + str(x)).json()
        Longitude.append(response['longitude'])
        Latitude.append(response['latitude'])
        LaunchSite.append(response['name'])

def getPayloadData(data):
    for load in data['payloads']:
        response = requests.get("https://api.spacexdata.com/v4/payloads/" + str(load)).json()
        PayloadMass.append(response['mass_kg'])
        Orbit.append(response['orbit'])

def getCoreData(data):
    for core in data['cores']:
        if core['core'] is not None:
            response = requests.get("https://api.spacexdata.com/v4/cores/" + str(core['core'])).json()
            Block.append(response['block'])
            ReusedCount.append(response['reuse_count'])
            Serial.append(response['serial'])
        else:
            Block.append(None); ReusedCount.append(None); Serial.append(None)
        Outcome.append(str(core['landing_success']) + ' ' + str(core['landing_type']))
        Flights.append(core['flight'])
        GridFins.append(core['gridfins'])
        Reused.append(core['reused'])
        Legs.append(core['legs'])
        LandingPad.append(core['landpad'])

getBoosterVersion(data)
getLaunchSite(data)
getPayloadData(data)
getCoreData(data)

### Assemble the final flat table

In [ ]:
launch_dict = {
    'FlightNumber': list(data['flight_number']),
    'Date': list(data['date']),
    'BoosterVersion': BoosterVersion,
    'PayloadMass': PayloadMass,
    'Orbit': Orbit,
    'LaunchSite': LaunchSite,
    'Outcome': Outcome,
    'Flights': Flights,
    'GridFins': GridFins,
    'Reused': Reused,
    'Legs': Legs,
    'LandingPad': LandingPad,
    'Block': Block,
    'ReusedCount': ReusedCount,
    'Serial': Serial,
    'Longitude': Longitude,
    'Latitude': Latitude
}

launch_df = pd.DataFrame(launch_dict)
launch_df = launch_df[launch_df['BoosterVersion'] == 'Falcon 9']
launch_df['FlightNumber'] = list(range(1, launch_df.shape[0] + 1))
launch_df.head()

### Fill missing payload mass values with the column mean, then save

In [ ]:
launch_df['PayloadMass'] = launch_df['PayloadMass'].fillna(launch_df['PayloadMass'].mean())
launch_df.to_csv('dataset_part_1.csv', index=False)
print("Saved", launch_df.shape[0], "rows to dataset_part_1.csv")

## Summary

This notebook pulls every past Falcon 9 launch from the public SpaceX API across four endpoints (launches, rockets, launchpads, payloads, cores), flattens the nested JSON into one row per launch, and writes the result to `dataset_part_1.csv`, which feeds directly into the data wrangling step.